In [38]:
import pandas as pd


imoex_df=pd.read_csv("../data/raw/imoex_daily.csv")
prices_df=pd.read_csv("../data/raw/prices_daily.csv")


In [2]:
sector_to_ticker_df=pd.read_csv("../config/universe.csv")

In [12]:
sector_to_ticker_df.rename({'secid': 'ticker'},axis=1,inplace=True)
sector_to_ticker_df.drop(['boardid','is_pref','snapshot_date','shortname'],axis=1,inplace=True)

In [13]:
sector_to_ticker_df.columns=['index', 'ticker', 'sector']

Index(['index', 'ticker', 'sector'], dtype='object')

In [5]:
sector_to_ticker_df['sector']=sector_to_ticker_df['sector'].apply(lambda x: x.lower())


In [6]:
imoex_df.rename(columns={"close":"close_ix"},inplace=True)
imoex_df.drop("index_code", axis=1, inplace=True)
imoex_df.set_index('trade_date',inplace=True)

In [40]:
prices_df.rename(columns={"secid":"ticker"},inplace=True)
prices_df.set_index('trade_date',inplace=True)

In [8]:
# prices_ix_df.to_csv('../data/processed/prices_clean_no_secs.csv')

In [9]:
prices_ix_df=pd.read_csv("../data/processed/prices_clean_no_secs.csv")

In [14]:
if 'trade_date' not in prices_ix_df.columns:
    prices_ix_df = prices_ix_df.reset_index()

# Нормализуем названия секторов
sector_to_ticker_df['sector'] = sector_to_ticker_df['sector'].str.lower().str.strip()

# 1. Привязываем сектор компании к акциям напрямую из universe.csv
# Сектор тикера известен всегда (для всех 44 тикеров и всех дат с 2018 года)
prices_ix_df = pd.merge(
    prices_ix_df,
    sector_to_ticker_df[['ticker', 'sector']].drop_duplicates(),
    on='ticker',
    how='left'
)


In [20]:
prices_ix_df.rename(columns={'sector_x': 'sector'}, inplace=True)

In [21]:
prices_ix_df.drop(columns=['sector_y'], inplace=True)

In [30]:
#Добавим сектор 'строительство' для тикеров MOEXRE, так как в universe.csv они не указаны.
MOEXRE=['PIKK', 'SMLT', 'LSRG', 'ETLN', 'APRI', 'MSST']
prices_ix_df.loc[prices_ix_df['ticker'].isin(MOEXRE), 'sector'] = 'строительство'

In [35]:
prices_ix_df['ticker'].unique()

array(['AFKS', 'AFLT', 'ALRS', 'BSPB', 'CBOM', 'CHMF', 'GAZP', 'GMKN',
       'IRAO', 'LKOH', 'MAGN', 'MOEX', 'MTSS', 'NLMK', 'NVTK', 'PHOR',
       'PLZL', 'ROSN', 'RTKM', 'RUAL', 'SBER', 'SBERP', 'SNGS', 'SNGSP',
       'TATN', 'TATNP', 'TRNFP', 'VTBR', 'ENPG', 'FLOT', 'MDMG', 'OZON',
       'RENI', 'VKCO', 'POSI', 'UGLD', 'SVCB', 'YDEX', 'HEAD', 'T', 'X5',
       'CNRU', 'RAGR', 'DOMRF'], dtype=object)

In [31]:
# Проверяем: сектор заполнен для 100% строк (0 пропусков)
print('Пропусков в sector:', prices_ix_df['sector'].isna().sum())
print('Количество уникальных тикеров:', prices_ix_df['ticker'].nunique())
prices_ix_df.info()


Пропусков в sector: 0
Количество уникальных тикеров: 44
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76843 entries, 0 to 76842
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   trade_date  76843 non-null  object 
 1   close_ix    76843 non-null  float64
 2   boardid     76843 non-null  object 
 3   ticker      76843 non-null  object 
 4   open        76605 non-null  float64
 5   high        76605 non-null  float64
 6   low         76605 non-null  float64
 7   close       76605 non-null  float64
 8   volume      76843 non-null  int64  
 9   value       76843 non-null  float64
 10  numtrades   76843 non-null  int64  
 11  sector      76843 non-null  object 
dtypes: float64(6), int64(2), object(4)
memory usage: 7.0+ MB


In [42]:
prices_ix_df.to_csv('../data/processed/prices_ix_secs.csv', index=False)


In [33]:
prices_ix_df.groupby('sector')['ticker'].unique()

sector
информационные технологии                 [OZON, VKCO, POSI, YDEX, HEAD, CNRU]
металлы и добыча             [ALRS, CHMF, GMKN, MAGN, NLMK, PLZL, RUAL, ENP...
нефть и газ                  [GAZP, LKOH, NVTK, ROSN, SNGS, SNGSP, TATN, TA...
потребительский сектор                                        [MDMG, X5, RAGR]
телекоммуникации                                                  [MTSS, RTKM]
транспорт                                                         [AFLT, FLOT]
финансы                      [AFKS, BSPB, CBOM, MOEX, SBER, SBERP, VTBR, RE...
химия и нефтехимия                                                      [PHOR]
электроэнергетика                                                       [IRAO]
Name: ticker, dtype: object